 # NBA 디트로이트 팀 성과 분석 실습



 ## 학습 목표

 - "문제 → 검색 → 적용 → 검증" 프로세스 실습

 - AI 답변을 그대로 복붙하지 않고 단계별로 이해하며 구현

 - 같은 결과를 얻는 다양한 방법 탐색

 - 직접 검증을 통한 결과 확인



 ## 분석 목표

 1. 디트로이트 팀의 전체 게임 승률과 리그 평균 승률 비교

 2. 디트로이트 팀의 클러치 게임 승률과 하위 5개 팀 승률 비교

 3. 분석 결과 시각화

 ---

 # 실습 1: NBA 디트로이트 팀 성과 분석

 ---

 ## 1단계: 문제 정의 및 분해

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import platform

# OS 자동 감지 및 한글 폰트 설정
system = platform.system()

if system == 'Windows':
    # Windows: 맑은 고딕 사용
    plt.rcParams['font.family'] = 'Malgun Gothic'
    
elif system == 'Darwin':  # Darwin = macOS
    # macOS: 애플 고딕 사용
    plt.rcParams['font.family'] = 'AppleGothic'
    
elif system == 'Linux':
    # Linux: 나눔 고딕 사용 (사전 설치 필요)
    # 터미널: sudo apt-get install -y fonts-nanum
    plt.rcParams['font.family'] = 'NanumGothic'
    
else:
    print(f"알 수 없는 OS: {system}")

# 모든 OS 공통: 음수 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

print(f"OS: {system}, 설정된 폰트: {plt.rcParams['font.family']}")

 ### 문제 1-1: 분석 문제를 논리적 단계로 분해하기



 "디트로이트 팀의 성과 분석"이라는 큰 문제를 해결 가능한 작은 단계로 나누세요.

In [ ]:
# 전체 분석 단계
"""
1단계: 데이터 로드 및 구조 파악
2단계: 데이터 전처리
3단계: 데이터 탐색
4단계: 모든 팀 승률 계산 (GroupBy, Pivot Table 방법)
5단계: 검증을 위한 디트로이트 팀 승률 직접 계산
6단계: 전체 승률 비교 시각화
7단계: 클러치 게임 식별 및 분석 (과제)
8단계: 클러치 게임 결과 시각화 (과제)
"""

 ## 2단계: 데이터 로드 및 전처리

 ### 문제 2-1: 데이터 로드

In [ ]:
# 필요한 데이터 로드
games = pd.read_csv("games.csv")
teams = pd.read_csv("teams.csv")

print("데이터 로드 완료")
print(f"games shape: {games.shape}")
print(f"teams shape: {teams.shape}")

 ### 문제 2-2: 데이터 구조 파악

In [ ]:
# games 데이터의 정보 확인
print("Games 데이터 정보:")
print(games.info())
print("\n첫 5행:")
display(games.head())


In [ ]:
# teams 데이터의 정보 확인
print("Teams 데이터 정보:")
print(teams.info())
print("\n첫 5행:")
display(teams.head())


 ### 문제 2-3: 데이터 품질 확인

 - 결측치, 중복 데이터 확인

In [ ]:
# 결측치 확인
print("=== Games 데이터 결측치 확인 ===")
print(games.isnull().sum())
print(f"\n전체 행 수: {len(games)}")

print("\n=== Teams 데이터 결측치 확인 ===")
print(teams.isnull().sum())
print(f"\n전체 행 수: {len(teams)}")


In [ ]:
# 중복 데이터 확인
print("=== 중복 데이터 확인 ===")
# 고유값 GAME_ID 기준으로 확인
duplicate_games = games.duplicated(subset=['GAME_ID']).sum()
print(f"Games 데이터 중복 행: {duplicate_games}")

# 고유값 TEAM_ID 기준으로 확인
duplicate_teams = teams.duplicated(subset=['TEAM_ID']).sum()
print(f"Teams 데이터 중복 행: {duplicate_teams}")


 ### 문제 2-4: 데이터 전처리

In [ ]:
# Step 1: 중복 제거
before_rows = len(games)
games = games.drop_duplicates(subset=['GAME_ID'])
after_rows = len(games)
print(f"제거된 중복 행: {before_rows - after_rows}")


In [ ]:
# Step 2: 결측치 처리 - 승패 정보가 없는 게임 확인
# HOME_TEAM_WINS가 결측인 경우 확인
missing_wins = games[games['HOME_TEAM_WINS'].isnull()]
print(f"승패 정보가 없는 게임: {len(missing_wins)}개")


In [ ]:
# Step 3: 점수 데이터 확인 및 처리
# 점수 컬럼 결측치 확인
score_missing = games[['PTS_home', 'PTS_away']].isnull().sum()
print("점수 결측치:")
print(score_missing)

# 점수가 없는 게임 확인
games_no_score = games[(games['PTS_home'].isnull()) | (games['PTS_away'].isnull())]
print(f"\n점수가 없는 게임: {len(games_no_score)}개")

if len(games_no_score) > 0:
    # 클러치 게임 분석을 위해 점수가 필요하므로 해당 게임 제거
    games_with_score = games.dropna(subset=['PTS_home', 'PTS_away'])
    print(f"점수가 있는 게임만 필터링: {len(games_with_score)}개")
    games = games_with_score


 ### 문제 2-5: 파생 변수 생성

In [ ]:
# Step 1: games테이블의 팀 이름에 대한 컬럼 생성
# - teams 테이블의 도시명 활용

 ## 3단계: 데이터 탐색

 ### 문제 3-1: 주요 컬럼 이해하기

 ## 4단계: 모든 팀 승률 계산 (GroupBy, Pivot Table)

- 4-1: GroupBy 
    - 직접 단계적 풀이 도출 또는 AI 검색으로 도출

- 4-2: Pivot Table (과제)
    - 직접 단계적 풀이 도출 또는 AI 검색으로 도출

 ### 문제 4-1: GroupBy를 사용한 승률 계산

In [ ]:
# Step 1: 홈팀 기준 집계


In [ ]:
# Step 2: 원정팀 기준 집계


In [ ]:
# Step 3: 홈과 원정 통합


In [ ]:
# Step 4: 정렬


 ### 문제 4-2: Pivot Table을 사용한 승률 계산 (제공)

In [ ]:
# Step 1: 홈 게임 집계
home_stats = games.pivot_table(
    index='HOME_TEAM',
    values='HOME_TEAM_WINS',
    aggfunc=['sum', 'count']
)
home_stats.columns = ['home_wins', 'home_games']
print("홈 게임 집계 완료")

# Step 2: 원정 게임 집계 (승패 반전)
games['AWAY_WINS'] = 1 - games['HOME_TEAM_WINS']
away_stats = games.pivot_table(
    index='VISITOR_TEAM',
    values='AWAY_WINS',
    aggfunc=['sum', 'count']
)
away_stats.columns = ['away_wins', 'away_games']
print("원정 게임 집계 완료")

# Step 3: 인덱스 기준 병합
team_stats_pivot = home_stats.join(away_stats, how='outer').fillna(0)
team_stats_pivot['total_wins'] = team_stats_pivot['home_wins'] + team_stats_pivot['away_wins']
team_stats_pivot['total_games'] = team_stats_pivot['home_games'] + team_stats_pivot['away_games']
team_stats_pivot['winrate'] = team_stats_pivot['total_wins'] / team_stats_pivot['total_games']

# 정렬 및 인덱스 리셋
team_stats_pivot = team_stats_pivot.sort_values('winrate', ascending=False).reset_index()
team_stats_pivot.rename(columns={'HOME_TEAM': 'team'}, inplace=True)
    
print("\n완료! 상위 3팀:")
display(team_stats_pivot.head(3))

 ### 문제 4-3: 두 방법 결과 비교

In [ ]:
# GroupBy와 Pivot Table 결과 비교
comparison = pd.merge(
    team_stats_df[['team', 'winrate']].rename(columns={'winrate': 'groupby_winrate'}),
    team_stats_pivot[['team', 'winrate']].rename(columns={'winrate': 'pivot_winrate'}),
    on='team'
)

comparison['difference'] = abs(comparison['groupby_winrate'] - comparison['pivot_winrate'])

print("두 방법의 승률 차이 확인:")
print(f"최대 차이: {comparison['difference'].max():.10f}")
print(f"평균 차이: {comparison['difference'].mean():.10f}")
print(f"모든 값이 동일: {comparison['difference'].max() < 1e-10}")

print("\n비교 샘플 (상위 5팀):")
display(comparison.head())

 ## 5단계: 검증을 위한 디트로이트 팀 승률 직접 계산

- 5단계: 검증을 위한 디트로이트 팀 승률 계산
    - 직접 도출

 ### 문제 5-1: 홈 게임 분석

In [ ]:
# Step 5-1-1: 디트로이트가 홈팀인 게임 찾기


In [ ]:
# Step 5-1-2: 홈 게임 승리 수 계산


In [ ]:
# Step 5-1-3: 홈 승률 계산


 ### 문제 5-2: 원정 게임 분석

In [ ]:
# Step 5-2-1: 디트로이트가 원정팀인 게임 찾기


In [ ]:
# Step 5-2-2: 원정 게임 승패 판단


In [ ]:
# Step 5-2-3: 원정 승률 계산

 ### 문제 5-3: 전체 승률 계산

In [ ]:
# Step 5-3-1: 홈과 원정 통합


In [ ]:
# Step 5-3-2: 전체 승률 계산


In [ ]:
# Step 5-3-3: 검증 - GroupBy 결과와 비교


 ### 문제 5-4: 리그 내 위치 분석

In [ ]:
# Step 5-4-1: 리그 평균 승률 계산


In [ ]:
# Step 5-4-2: 디트로이트 순위 확인


In [ ]:
# Step 5-4-3: 디트로이트와 리그 평균 비교


 ## 6단계: 전체 승률 비교 시각화

 ### 문제 6-1: 시각화 데이터 준비

In [ ]:
# Step 6-1-1: 필요한 데이터 추출


In [ ]:
# Step 6-1-2: 그래프용 데이터 구성


 ### 문제 6-2: 막대 그래프 생성

In [ ]:
# 그래프 생성
plt.figure(figsize=(12, 6))

# 색상 설정 (디트로이트 강조)
colors = ['#FF4444',  # 디트로이트 - 빨간색
          '#666666',  # 리그 평균 - 회색
          '#4CAF50', '#4CAF50', '#4CAF50',  # 상위팀 - 초록색
          '#FFC107', '#FFC107', '#FFC107']  # 하위팀 - 노란색

# 막대 그래프 생성
bars = plt.bar(teams_labels, winrates, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

# 제목 및 레이블
plt.title('NBA 팀별 승률 비교 (디트로이트 vs 리그 평균 vs 상하위팀)', 
          fontsize=16, fontweight='bold', pad=20)
plt.ylabel('승률', fontsize=12)
plt.xlabel('팀', fontsize=12)

# 막대 위에 값 표시
for bar, rate in zip(bars, winrates):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{rate:.3f}\n({rate*100:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# 그래프 꾸미기
plt.grid(True, axis='y', alpha=0.3, linestyle='--')
plt.ylim(0, max(winrates) * 1.1)

# 범례 추가
legend_elements = [
    plt.Rectangle((0,0),1,1, color='#FF4444', alpha=0.8, label='디트로이트'),
    plt.Rectangle((0,0),1,1, color='#666666', alpha=0.8, label='리그 평균'),
    plt.Rectangle((0,0),1,1, color='#4CAF50', alpha=0.8, label='상위 3팀'),
    plt.Rectangle((0,0),1,1, color='#FFC107', alpha=0.8, label='하위 3팀')
]
plt.legend(handles=legend_elements, loc='upper right')

# x축 레이블 회전
plt.xticks(rotation=45, ha='right')

# 레이아웃 조정 및 표시
plt.tight_layout()
plt.show()

 ---

 # 과제

 ---

 ## 7단계: 클러치 게임 식별 및 분석 (과제)



 ### 과제 내용

 1. 클러치 게임 정의: 점수 차이가 5점 이하인 게임

 2. 디트로이트의 클러치 게임 승률 계산

 3. 모든 팀의 클러치 게임 승률 계산

 4. 디트로이트와 하위 5팀 비교



 ### 힌트

 - 점수 차이: `abs(games['PTS_home'] - games['PTS_away'])`

 - 조건부 승패 판단: `np.where()` 활용

 - 클러치 게임에서도 홈/원정 구분 필요

In [ ]:
# TODO: 7단계 과제를 구현하세요


 ## 8단계: 클러치 게임 결과 시각화 (과제)



 ### 과제 내용

 1. 클러치 승률 비교 막대 그래프 (디트로이트 vs 하위 5팀)



 ### 요구 사항

 - 디트로이트는 다른 색으로 강조

In [ ]:
# TODO: 8단계 과제를 구현하세요


 ## 분석 결과 요약 정리